In [1]:
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/europarl", "de-en")

print(ds)
print(ds["train"][:5])

small_ds = ds["train"].shuffle(seed=42).select(range(100))

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 1961119
    })
})
{'translation': [{'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}, {'de': 'Ich erkläre die am Freitag, dem 17. Dezember unterbrochene Sitzungsperiode des Europäischen Parlaments für wiederaufgenommen, wünsche Ihnen nochmals alles Gute zum Jahreswechsel und hoffe, daß Sie schöne Ferien hatten.', 'en': 'I declare resumed the session of the European Parliament adjourned on Friday 17 December 1999, and I would like once again to wish you a happy new year in the hope that you enjoyed a pleasant festive period.'}, {'de': 'Wie Sie feststellen konnten, ist der gefürchtete "Millenium-Bug " nicht eingetreten. Doch sind Bürger einiger unserer Mitgliedstaaten Opfer von schrecklichen Naturkatastrophen geworden.', 'en': "Although, as you will have seen, the dreaded 'millennium bug' failed to materialise, still the people in a number of countries suffered a series o

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer

tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-de")

model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-de")

/Users/katharinazeilnhofer/VS Code/machine-learning-project/.venv/lib/python3.10/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [3]:
source_lang = "de"
target_lang = "en"
prefix = "translate German to English: "


def preprocess_function(examples):
    inputs = [prefix + example[source_lang] for example in examples["translation"]]
    targets = [example[target_lang] for example in examples["translation"]]
    model_inputs = tokenizer(inputs, max_length=128, truncation=True)

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

small_tokenized_ds = small_ds.map(preprocess_function, batched=True)

In [4]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [5]:
split_ds = small_tokenized_ds.train_test_split(test_size=0.2)

In [8]:
import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.array(preds)
    if preds.ndim == 3:
        preds = preds[:, 0, :]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

In [9]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

training_args = Seq2SeqTrainingArguments(
    logging_strategy="steps",
    logging_steps=5,
    report_to="none",
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=False,
    dataloader_pin_memory=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=split_ds["train"],
    eval_dataset=split_ds["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [10]:
trainer.train()

{'loss': 4.5107, 'grad_norm': 7.562136650085449, 'learning_rate': 0.0, 'epoch': 1.0}
{'eval_loss': 4.101400375366211, 'eval_bleu': 0.6303123308473878, 'eval_runtime': 7.7816, 'eval_samples_per_second': 2.57, 'eval_steps_per_second': 0.257, 'epoch': 1.0}
{'train_runtime': 12.731, 'train_samples_per_second': 6.284, 'train_steps_per_second': 0.393, 'train_loss': 4.510666275024414, 'epoch': 1.0}


TrainOutput(global_step=5, training_loss=4.510666275024414, metrics={'train_runtime': 12.731, 'train_samples_per_second': 6.284, 'train_steps_per_second': 0.393, 'train_loss': 4.510666275024414, 'epoch': 1.0})

In [12]:
import pandas as pd

results = trainer.train()

df = pd.DataFrame(trainer.state.log_history)

train_logs = df[df["loss"].notna()][["epoch", "loss"]]
eval_logs = df[df["eval_loss"].notna()][["epoch", "eval_loss", "eval_bleu"]]

result_df = train_logs.merge(eval_logs, on="epoch", how="outer")
result_df.style.format({"loss": "{:.4f}", "eval_loss": "{:.4f}", "eval_bleu": "{:.2f}"})

{'loss': 3.9097, 'grad_norm': 6.646390438079834, 'learning_rate': 0.0, 'epoch': 1.0}
{'eval_loss': 3.818643808364868, 'eval_bleu': 0.8275848499045719, 'eval_runtime': 27.3792, 'eval_samples_per_second': 0.73, 'eval_steps_per_second': 0.073, 'epoch': 1.0}
{'train_runtime': 31.5677, 'train_samples_per_second': 2.534, 'train_steps_per_second': 0.158, 'train_loss': 3.9096553802490233, 'epoch': 1.0}


,epoch,loss,eval_loss,eval_bleu
0,1.000000,3.9097,3.8186,0.83


In [15]:
test_sentences = [
    "I study Multilingual Technologies in Vienna.",
    "The weather is very nice today.",
    "Can you please help me?",
    "The cat sits on the table.",
]

for text in test_sentences:
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs)
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"EN: {text}")
    print(f"DE: {translation}\n")

EN: I study Multilingual Technologies in Vienna.
DE: Ich studiere Mehrsprachige Technologien in Wien.

EN: The weather is very nice today.
DE: Das Wetter ist heute sehr schön.

EN: Can you please help me?
DE: Können Sie mir bitte helfen?

EN: The cat sits on the table.
DE: Die Katze sitzt auf dem Tisch.

